# Disaster Triage: 3-Class RED / YELLOW / GREEN (`models/train_disaster_triage_3class.ipynb`)

Trains the model the TriageBox node actually needs, against the label it actually
emits. Independent of the notebooks under `models/` -- nothing here reads or writes
their artefacts.

### Why not 5-class ESI

Exact 5-level ESI accuracy on this dataset tops out near 69% (68% reported for the
same Yale 560k cohort in the literature). It is not a 90% task: ESI is a nurse's
judgement over chief complaint, mental status and predicted resource need, and the
vitals explain only part of it. Measured on one fixed split:

| features | 5-class accuracy |
| --- | --- |
| majority class (always ESI 3) | 43.2% |
| 15 vitals features (what the old pipeline used) | 48.8% |
| chief complaints only, **no vitals at all** | 65.1% |
| vitals + 200 chief-complaint flags | 67.0% |
| + arrival / history context | 69.4% |

The ~50% plateau was a missing-feature ceiling, not an irreducible class overlap:
`k`-NN(100) local purity in the 15-feature vitals space is 0.481, so no model could
have beaten ~48.5% there. The dataset carries 200 `cc_*` columns and the old
pipeline used one.

### Why undertriage, not accuracy

START/SALT doctrine sets the bar as **undertriage < 5%**, with overtriage tolerated
up to ~50%. Accuracy scores "RED called GREEN" the same as "GREEN called YELLOW";
only the first one kills someone. So the RED threshold is tuned for sensitivity on
the validation set and the holdout reports the safety pair.

`BLACK` is deliberately not a class here -- no pulse and no respiration is a
firmware rule, not a statistical decision.

In [ ]:
import json, os, pickle
import numpy as np, pandas as pd, lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score,
                             roc_auc_score, confusion_matrix, cohen_kappa_score)

ROOT = '..' if os.path.basename(os.getcwd()) == 'models' else '.'
COLS = json.load(open(f'{ROOT}/val_columns.json'))

# Measured by the node. NO BLOOD PRESSURE: TB_FLAG_BP_VALID is permanently clear in
# triagebox-station (main/lora_vital.h), so bp_sys/bp_dia would arrive as constant
# zeros. Dropping BP costs 0.2 accuracy points once chief complaints are present --
# it only looked important before because nothing better was in the feature set.
#
# The *_min / *_max pairs are ED-stay aggregates in this dataset but a rolling
# window on the node. Expect some distribution shift on real hardware.
MEASURED = ['triage_vital_hr', 'triage_vital_rr', 'triage_vital_o2',
            'pulse_min', 'resp_min', 'spo2_min',
            'pulse_max', 'resp_max', 'spo2_max']

# Entered by the responder. triage_vital_temp is here, not in MEASURED: the node has
# no thermometer (tb_regs.h snapshot carries hr/spo2/rr/bp/battery only). Leave it
# NaN when unmeasured -- LightGBM splits on NaN natively.
MANUAL = ['age', 'gender', 'triage_vital_temp']

# 200 one-hot chief-complaint flags. Mean set per visit is 1.13 and 99.5% of visits
# have at least one, so in the UI this is ONE searchable dropdown, not 200
# checkboxes. These flags are what buys the +20 accuracy points.
CC = [c for c in COLS if c.startswith('cc_')]

FEATURES = MANUAL + MEASURED + CC
LABELS = ['RED', 'YELLOW', 'GREEN']          # ESI 1-2 / ESI 3 / ESI 4-5
TARGET_RED_SENSITIVITY = 0.90

# ESP32-S3 budget. 27.9k nodes is ~327 KB as a flat const array in .rodata; do not
# transpile this with m2cgen -- nested if/else in C balloons to a file the compiler
# chokes on. Store {int16 feature, float threshold, int16 left, int16 right} and
# walk it with a ~30-line interpreter.
PARAMS = dict(objective='multiclass', num_class=3, learning_rate=0.05, num_leaves=31,
              feature_fraction=0.7, bagging_fraction=0.8, bagging_freq=1,
              min_child_samples=50, n_estimators=300, verbosity=-1,
              random_state=42, n_jobs=-1)

print(f'{len(FEATURES)} features = {len(MANUAL)} manual + {len(MEASURED)} measured '
      f'+ {len(CC)} chief complaints')

In [ ]:
# ---------------------------------------------------------------------------
# Load. No na.omit: the old pipeline dropped every row with any NA across 15
# columns, discarding 70.5% of the dataset (560,486 -> 165,240) mostly for a
# missing triage_vital_o2, which is absent in 48.4% of visits. LightGBM learns a
# default direction for NaN, so those 395k rows are free training data.
# ---------------------------------------------------------------------------
df = pd.read_csv(f'{ROOT}/datasets/5v_raw.csv', usecols=['esi'] + FEATURES,
                 low_memory=False)
df = df[df['esi'].notna()].copy()

df['gender'] = (df['gender'].astype(str) == 'Male').astype(np.int8)
for c in CC:
    df[c] = pd.to_numeric(df[c], errors='coerce').fillna(0).astype(np.int8)
for c in [c for c in MANUAL if c != 'gender'] + MEASURED:
    df[c] = pd.to_numeric(df[c], errors='coerce')

esi = df['esi'].astype(int).values
y = np.where(esi <= 2, 0, np.where(esi == 3, 1, 2))
X = df[FEATURES].values.astype(np.float32)

itr, itmp = train_test_split(np.arange(len(y)), test_size=0.30, stratify=y,
                             random_state=42)
iva, ite = train_test_split(itmp, test_size=0.50, stratify=y[itmp], random_state=42)

print(f'rows={len(y)}  train={len(itr)}  val={len(iva)}  test={len(ite)}')
for name, n in zip(LABELS, np.bincount(y)):
    print(f'  {name:<7}{n:>8}  {100 * n / len(y):.1f}%')
print(f'  NaN cells kept: {100 * np.isnan(X).mean():.1f}% of the matrix')

In [ ]:
# ---------------------------------------------------------------------------
# Train. One plain multiclass model -- no SMOTE, no class weighting, no
# hierarchy, no stacking. Measured on this data those add nothing: a plain
# LightGBM matched the 4-layer SMOTE+stacking pipeline to within 0.3 points, and
# class_weight='balanced' on a meta-learner cost 21 points by forcing ESI 1
# (0.13% of rows) to be predicted as often as ESI 3 (43%).
# ---------------------------------------------------------------------------
model = lgb.LGBMClassifier(**PARAMS)
model.fit(X[itr], y[itr], eval_set=[(X[iva], y[iva])],
          callbacks=[lgb.early_stopping(40, verbose=False)])

dump = model.booster_.dump_model()
n_nodes = sum(t['num_leaves'] for t in dump['tree_info'])
print(f'best_iteration={model.best_iteration_}  trees={len(dump["tree_info"])}  '
      f'nodes={n_nodes}  ~{n_nodes * 16 / 1024:.0f} KB in .rodata')

p_val = model.predict_proba(X[iva])
p_test = model.predict_proba(X[ite])
argmax = p_test.argmax(1)
print(f'\nplain argmax   acc={accuracy_score(y[ite], argmax):.4f}   '
      f'balanced={balanced_accuracy_score(y[ite], argmax):.4f}   '
      f'macroF1={f1_score(y[ite], argmax, average="macro"):.4f}   '
      f'AUC={roc_auc_score(y[ite], p_test, multi_class="ovr", average="macro"):.4f}')

In [ ]:
# ---------------------------------------------------------------------------
# Threshold. Call RED whenever p(RED) clears t; below it, YELLOW vs GREEN on
# their own argmax. Searched on validation only -- the holdout never sees it.
# ---------------------------------------------------------------------------
def apply_threshold(p, t):
    return np.where(p[:, 0] >= t, 0, np.where(p[:, 1] >= p[:, 2], 1, 2))


def safety(y_true, y_pred):
    c = confusion_matrix(y_true, y_pred, labels=[0, 1, 2])
    # Overtriage is measured over EVERY non-RED casualty, not just GREEN. Counting
    # only GREEN->RED flatters the number badly: at the selected threshold it reads
    # 15% that way and 36% correctly, because most of the promoted cases are YELLOW.
    return dict(red_sensitivity=c[0, 0] / c[0].sum(),
                undertriage=c[0, 2] / c[0].sum(),
                overtriage=(c[1, 0] + c[2, 0]) / (c[1].sum() + c[2].sum()),
                overtriage_green_only=c[2, 0] / c[2].sum(),
                accuracy=accuracy_score(y_true, y_pred))


grid = np.round(np.arange(0.50, 0.02, -0.01), 2)
hit = [t for t in grid
       if safety(y[iva], apply_threshold(p_val, t))['red_sensitivity']
       >= TARGET_RED_SENSITIVITY]
THRESHOLD = float(hit[0]) if hit else 0.50
print(f'RED threshold chosen on validation: {THRESHOLD}')

print('\n  t      RED_sens  undertriage  overtriage  accuracy')
for t in sorted({0.50, 0.40, 0.30, 0.25, 0.20, THRESHOLD, 0.15, 0.10}, reverse=True):
    s = safety(y[ite], apply_threshold(p_test, t))
    tag = '  <-- selected' if t == THRESHOLD else ''
    print(f'  {t:<6.2f} {s["red_sensitivity"]:<9.4f} {s["undertriage"]:<12.4f} '
          f'{s["overtriage"]:<11.4f} {s["accuracy"]:.4f}{tag}')

In [ ]:
# ---------------------------------------------------------------------------
# Holdout report. These four numbers are what the proposal should quote.
# ---------------------------------------------------------------------------
pred = apply_threshold(p_test, THRESHOLD)
s = safety(y[ite], pred)
auc = roc_auc_score(y[ite], p_test, multi_class='ovr', average='macro')
cm = confusion_matrix(y[ite], pred, labels=[0, 1, 2])

print('HOLDOUT TEST -- 3-class disaster triage')
print(f'  RED sensitivity        {s["red_sensitivity"]:.4f}')
print(f'  undertriage RED->GREEN {s["undertriage"]:.4f}    doctrine: < 0.05')
print(f'  overtriage non-RED     {s["overtriage"]:.4f}    doctrine: <= 0.50')
print(f'    of which GREEN only  {s["overtriage_green_only"]:.4f}')
print(f'  accuracy at threshold  {s["accuracy"]:.4f}')
print(f'  accuracy at plain argmax {accuracy_score(y[ite], argmax):.4f}')
print(f'  macro AUC-ROC          {auc:.4f}')

import matplotlib.pyplot as plt

norm = cm / cm.sum(1, keepdims=True)
fig, ax = plt.subplots(figsize=(6.2, 5.2))
ax.imshow(norm, cmap='Greens', vmin=0, vmax=1)
for i in range(3):
    for j in range(3):
        ax.text(j, i, f'{cm[i, j]}\n({100 * norm[i, j]:.1f}%)', ha='center',
                va='center', color='white' if norm[i, j] > 0.55 else 'black')
ax.set_xticks(range(3), LABELS)
ax.set_yticks(range(3), LABELS)
ax.set_xlabel('Predicted')
ax.set_ylabel('True')
ax.set_title(f'Disaster triage holdout (RED threshold {THRESHOLD})')
plt.tight_layout()
os.makedirs(f'{ROOT}/plots', exist_ok=True)
plt.savefig(f'{ROOT}/plots/disaster_triage_confusion_matrix.png', dpi=200,
            bbox_inches='tight')
plt.show()

In [ ]:
# ---------------------------------------------------------------------------
# Secondary: the same features against the original 5-level ESI, so the report
# has a number comparable with the literature. This is NOT what the node emits.
# ---------------------------------------------------------------------------
P5 = dict(PARAMS, num_class=5)
m5 = lgb.LGBMClassifier(**P5)
m5.fit(X[itr], esi[itr] - 1, eval_set=[(X[iva], esi[iva] - 1)],
       callbacks=[lgb.early_stopping(40, verbose=False)])
p5 = m5.predict_proba(X[ite])
pr5 = p5.argmax(1) + 1

print('5-class ESI (secondary, for literature comparison)')
print(f'  accuracy    {accuracy_score(esi[ite], pr5):.4f}')
print(f'  balanced    {balanced_accuracy_score(esi[ite], pr5):.4f}')
print(f'  QWK         {cohen_kappa_score(esi[ite], pr5, weights="quadratic"):.4f}')
print(f'  within +-1  {np.mean(np.abs(esi[ite] - pr5) <= 1):.4f}')
print(f'  macro AUC   {roc_auc_score(esi[ite], p5, multi_class="ovr", average="macro"):.4f}')

In [ ]:
# ---------------------------------------------------------------------------
# Export. The manifest is the single source of truth for feature order: the C
# exporter and the dashboard input form both read it, so neither can drift from
# the trained model.
# ---------------------------------------------------------------------------
deploy = f'{ROOT}/deploy'
os.makedirs(deploy, exist_ok=True)

with open(f'{deploy}/disaster_triage_3class.pkl', 'wb') as f:
    pickle.dump(dict(model=model, threshold=THRESHOLD, features=FEATURES,
                     labels=LABELS), f)

manifest = dict(labels=LABELS, red_threshold=THRESHOLD, feature_order=FEATURES,
                manual_inputs=MANUAL, measured_inputs=MEASURED,
                chief_complaints=CC, n_nodes=n_nodes,
                best_iteration=int(model.best_iteration_),
                holdout={k: round(float(v), 4) for k, v in s.items()},
                holdout_macro_auc=round(float(auc), 4))
with open(f'{deploy}/disaster_triage_manifest.json', 'w') as f:
    json.dump(manifest, f, indent=2)

print(f'wrote {deploy}/disaster_triage_3class.pkl')
print(f'wrote {deploy}/disaster_triage_manifest.json')

In [ ]:
# ---------------------------------------------------------------------------
# Runnable check. These are the claims the proposal makes; if a retrain breaks
# one, this cell fails instead of the number quietly changing in the report.
# ---------------------------------------------------------------------------
assert s['red_sensitivity'] >= 0.90, f'RED sensitivity {s["red_sensitivity"]:.4f}'
assert s['undertriage'] < 0.05, f'undertriage {s["undertriage"]:.4f} breaks doctrine'
assert s['overtriage'] <= 0.50, f'overtriage {s["overtriage"]:.4f} breaks doctrine'
assert n_nodes * 16 <= 2 * 1024 * 1024, f'{n_nodes} nodes exceeds the flash budget'
assert len(FEATURES) == len(set(FEATURES)), 'duplicate feature name'
assert not [f for f in FEATURES if 'sbp' in f or 'dbp' in f or 'bp_' in f], \
    'blood pressure leaked back in -- the node has no BP sensor'
assert json.load(open(f'{deploy}/disaster_triage_manifest.json'))['feature_order'] \
    == FEATURES, 'manifest feature order does not match the trained model'
print('all checks passed')